# nb_02 — Silver : conformer, dédupliquer, valider, convertir

Une ligne par événement de main-d'œuvre.

1. **Dédupliquer** les `event_id` rejoués — conserver le dernier `ingest_ts`.
2. **Conformer** le code du pays de travail — `CORE_HR` émet des codes ISO-3, `PAYROLL` émet des codes ISO-2.
3. **Valider et mettre en quarantaine** — paie négative, employés orphelins, dates incorrectes, devise inconnue.
4. **Convertir** les montants de paie de la devise locale en **CAD** (la grille est en CAD).
5. Écrire `silver.workforce_event`.

> Remarque : de nombreux événements (congés, déploiements) ne comportent légitimement **aucun montant**. Nous
> mettons en quarantaine la paie *négative*, et non la paie nulle.

## Charger les entrées Bronze

**Résumé.** Crée le schéma `silver` et charge les tables Bronze utilisées par ce notebook : les événements bruts (y compris les rejoués), ainsi que les tables de référence workers, workers-delta et FX.

<details>
<summary>Détails ligne par ligne</summary>

- `from pyspark.sql import functions as F, Window as W` — les assistants utilisés pour la déduplication et la mise en conformité.
- `CREATE SCHEMA IF NOT EXISTS silver` — s'assurer que le schéma Silver existe.
- `raw`, `wk`, `wkd`, `fx` — références à `bronze.workforce_events_raw`, `bronze.workers`, `bronze.workers_delta` et `bronze.fx_rates`.
- `print(...)` — affiche le nombre d'événements bruts (qui inclut encore les rejoués).

</details>

In [ ]:
from pyspark.sql import functions as F, Window as W

spark.sql("CREATE SCHEMA IF NOT EXISTS silver")
raw = spark.table("bronze.workforce_events_raw")
wk = spark.table("bronze.workers")
wkd = spark.table("bronze.workers_delta")
fx = spark.table("bronze.fx_rates")
print(f"bronze events (raw, incl replays): {raw.count():,}")

## 1. Dédupliquer les rejoués

**Résumé.** Ne conserve que la copie la plus récente de chaque `event_id`, en supprimant les doublons rejoués par le système de paie.

<details>
<summary>Détails ligne par ligne</summary>

- `W.partitionBy("event_id").orderBy(F.col("ingest_ts").desc())` — une fenêtre par événement, avec l'ingestion la plus récente en premier.
- `withColumn("_rn", F.row_number().over(w)).filter("_rn=1").drop("_rn")` — numérote les lignes de chaque événement et conserve uniquement le rang 1 (le plus récent).
- `print(...)` — affiche le nombre de lignes conservées et le nombre de doublons supprimés.

</details>

In [ ]:
w = W.partitionBy("event_id").orderBy(F.col("ingest_ts").desc())

dedup = raw.withColumn("_rn", F.row_number().over(w)).filter("_rn=1").drop("_rn")

print(f"after dedup: {dedup.count():,}  (removed {raw.count()-dedup.count():,})")

## 2. Mettre le code du pays de travail au format ISO-3

**Résumé.** Normalise le code du pays de travail afin que les deux systèmes sources concordent : `CORE_HR` envoie déjà des codes ISO-3, tandis que `PAYROLL` envoie des codes ISO-2, convertis par une petite map statique.

<details>
<summary>Détails ligne par ligne</summary>

- `spark.createDataFrame([...], ["_i2","_i3"])` — une petite référence ISO-2 vers ISO-3 pour les pays concernés.
- `withColumn("wcc3", F.when(F.length(...)==3, ...))` — conserve les valeurs qui sont déjà au format ISO-3.
- `.join(iso, dedup.work_country_code==iso._i2, "left")` — recherche le code ISO-3 correspondant aux valeurs ISO-2.
- `withColumn("work_country_iso3", F.coalesce("wcc3","_i3"))` — privilégie la valeur déjà au format ISO-3, sinon utilise la valeur mappée ; les colonnes auxiliaires sont supprimées.

</details>

In [ ]:
iso = spark.createDataFrame(
    [
        ("CA", "CAN"),
        ("US", "USA"),
        ("GB", "GBR"),
        ("FR", "FRA"),
        ("DE", "DEU"),
        ("JP", "JPN"),
        ("AU", "AUS"),
        ("SG", "SGP"),
        ("BR", "BRA"),
        ("AE", "ARE"),
        ("KE", "KEN"),
        ("IN", "IND"),
        ("MX", "MEX"),
    ],
    ["_i2", "_i3"],
)

conf = (
    dedup.withColumn(
        "wcc3", F.when(F.length("work_country_code") == 3, F.col("work_country_code"))
    )
    .join(iso, dedup.work_country_code == iso._i2, "left")
    .withColumn("work_country_iso3", F.coalesce("wcc3", "_i3"))
    .drop("_i2", "_i3", "wcc3")
)

## 3. Valider et mettre en quarantaine

| Règle | Motif |
|------|--------|
| `amount_local < 0` | `negative_pay` |
| `employee_id` ne correspond à aucun employé | `orphan_employee` |
| `event_date` est antérieure au 2021-01-01 ou future | `date_out_of_range` |
| devise inconnue **ou** pays non résolu | `unresolved_reference` |

Les montants nuls sont **valides** (événements hors paie). Les lignes en échec sont envoyées en quarantaine.

**Résumé.** Signale les lignes qui enfreignent les règles de qualité des données (paie négative, employés orphelins, dates hors plage, devise ou pays non résolu), écrit les échecs dans une table de quarantaine et conserve les lignes propres.

<details>
<summary>Détails ligne par ligne</summary>

- `valid_emp` / `valid_ccy` — ensembles d'identifiants d'employés connus (workers et workers-delta) et de devises connues, collectés une seule fois.
- La chaîne `F.when(...).when(...)` — attribue le premier `dq_reason` correspondant : `negative_pay`, `orphan_employee`, `date_out_of_range` ou `unresolved_reference`. Les montants nuls sont considérés comme valides.
- `quarantine = flagged.filter("dq_reason IS NOT NULL")` et `clean = flagged.filter("dq_reason IS NULL").drop("dq_reason")` — sépare les lignes en échec des lignes propres.
- `quarantine.write ... saveAsTable("silver.workforce_event_quarantine")` — conserve les échecs pour examen.
- `groupBy("dq_reason").count()` et `print` résument ce qui a été mis en quarantaine et le nombre de lignes conservées.

</details>

In [ ]:
valid_emp = set(
    r.employee_id
    for r in wk.select("employee_id")
    .union(wkd.select("employee_id"))
    .distinct()
    .collect()
)


valid_ccy = set(r.currency for r in fx.select("currency").distinct().collect())


flagged = conf.withColumn(
    "dq_reason",
    F.when(F.col("amount_local") < 0, "negative_pay")
    .when(~F.col("employee_id").isin(list(valid_emp)), "orphan_employee")
    .when(
        (F.col("event_date") < F.lit("2021-01-01"))
        | (F.col("event_date") > F.current_date()),
        "date_out_of_range",
    )
    .when(
        F.col("local_currency").isNull()
        | ~F.col("local_currency").isin(list(valid_ccy))
        | F.col("work_country_iso3").isNull(),
        "unresolved_reference",
    ),
)


quarantine = flagged.filter("dq_reason IS NOT NULL")


clean = flagged.filter("dq_reason IS NULL").drop("dq_reason")


(
    quarantine.write.format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("silver.workforce_event_quarantine")
)


quarantine.groupBy("dq_reason").count().orderBy("dq_reason").show()


print(f"clean rows continuing: {clean.count():,}")

## 4. Convertir la paie en CAD

La grille de paie est en CAD, mais les bureaux internationaux paient dans leur devise locale. Convertir
`amount_local` → `amount_cad` sur `(rate_month, currency)`. Les événements hors paie conservent un
montant nul.

**Résumé.** Convertit les montants en devise locale en CAD à l'aide du taux FX du mois de l'événement, sélectionne les colonnes conformes et écrit la table fiable `silver.workforce_event`.



<details>

<summary>Détails ligne par ligne</summary>



- `fx_lkp` — la table FX associée à `_rm` (mois du taux), `_ccy` (devise) et `cad_per_unit`.

- `withColumn("rate_month", F.date_format("event_date","yyyy-MM"))` — la clé de jointure correspondant au mois.

- `.join(fx_lkp, (rate_month==_rm) & (local_currency==_ccy), "left")` — associe le taux correspondant.

- `withColumn("amount_cad", F.when(amount_local isNull, None).otherwise(round(amount_local * cad_per_unit, 2)))` — une valeur nulle reste nulle ; sinon, convertit et arrondit. Un taux FX absent pour un événement rémunéré déclenche une erreur explicite.

- Le `.select(...)` final convertit les types et ordonne les colonnes de sortie (`amount_local`/`amount_cad` au format `decimal(18,2)`).

- `write ... saveAsTable("silver.workforce_event")` et le `print`/`show` confirment le résultat.



</details>


In [ ]:
fx_lkp = fx.select(
    F.col("rate_month").alias("_rm"), F.col("currency").alias("_ccy"), "cad_per_unit"
)

silver = (
    clean.withColumn("rate_month", F.date_format("event_date", "yyyy-MM"))
    .join(
        fx_lkp,
        (F.col("rate_month") == F.col("_rm"))
        & (F.col("local_currency") == F.col("_ccy")),
        "left",
    )
    .withColumn(
        "amount_cad",
        F.when(F.col("amount_local").isNull(), None).otherwise(
            F.round(F.col("amount_local") * F.col("cad_per_unit"), 2)
        ),
    )
    .select(
        "event_id",
        F.to_date("event_date").alias("event_date"),
        "employee_id",
        "cost_center_id",
        "classification_group",
        F.col("classification_level").cast("int").alias("classification_level"),
        "event_type",
        F.col("amount_local").cast("decimal(18,2)").alias("amount_local"),
        "local_currency",
        "work_country_iso3",
        F.col("amount_cad").cast("decimal(18,2)").alias("amount_cad"),
        "source_system",
        F.to_timestamp("ingest_ts").alias("ingest_ts"),
    )
)


missing_fx = silver.filter(
    F.col("amount_local").isNotNull() & F.col("amount_cad").isNull()
).count()

if missing_fx:

    raise ValueError(
        f"Missing FX rate for {missing_fx:,} paid event(s); check bronze.fx_rates."
    )


(
    silver.write.format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("silver.workforce_event")
)


print(f"silver.workforce_event: {silver.count():,} rows")


silver.show(5, truncate=False)